# Prefix-Conditioned Suffix Scoring（Revised, Teacher-Forced）

本版本针对“信号太弱”做了三项最小改进（仍不改仓库源码）：
1. 使用更强 corruption 候选，并自动选择 clean->corrupted drop 最大的方案。
2. 保持 target suffix 不变，只破坏 prefix 条件。
3. 对多个 patch site 做小范围 sweep，选择 recovery gain 最强的 site。

checkpoint 固定路径：
- /home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/standard/ckpt_standard_last.pt
- /home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/attnres/ckpt_attnres_last.pt


In [ ]:
# Cell 1. 环境与导入
import json
import math
import sys
from copy import deepcopy
from pathlib import Path

import torch
import matplotlib.pyplot as plt

CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / 'toygpt2').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('未找到仓库根目录（应包含 toygpt2 目录）')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.evaluate import load_checkpoint
from data.data import build_dataloaders
from interp.patching import patch_from_cache

print('REPO_ROOT:', REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)


In [ ]:
# Cell 2. 加载 checkpoint 和模型（固定服务器路径）
device = torch.device('cpu')
CHECKPOINTS = {
    'standard': str(REPO_ROOT / 'toygpt2_runs' / 'tinystories_dual' / 'standard' / 'ckpt_standard_last.pt'),
    'attnres': str(REPO_ROOT / 'toygpt2_runs' / 'tinystories_dual' / 'attnres' / 'ckpt_attnres_last.pt'),
}

MODEL_VARIANT = 'standard'  # 'standard' / 'attnres'
if MODEL_VARIANT not in CHECKPOINTS:
    raise ValueError(f"MODEL_VARIANT 必须是 {list(CHECKPOINTS.keys())}")

ckpt_path = Path(CHECKPOINTS[MODEL_VARIANT])
model, experiment, checkpoint = load_checkpoint(ckpt_path, device=device)
model = model.to(device).eval()

print('model_variant:', MODEL_VARIANT)
print('checkpoint:', ckpt_path)
print('model_type:', checkpoint.get('model_type'))
print('step:', checkpoint.get('step'))
print('n_layer:', experiment.model.n_layer)
print('n_head:', experiment.model.n_head)
print('n_embd:', experiment.model.n_embd)
print('vocab_size:', experiment.model.vocab_size)
print('block_size:', experiment.model.block_size)
print('device:', next(model.parameters()).device)


In [ ]:
# Cell 3. 从 TinyStories 抽样（与 checkpoint 数据配置对齐）
sample_model_config = deepcopy(experiment.model)
sample_model_config.block_size = int(min(48, experiment.model.block_size))

tiny_cfg = deepcopy(experiment.data)
tiny_cfg.dataset_type = 'tinystories'
tiny_cfg.block_stride = max(1, sample_model_config.block_size)
if tiny_cfg.train_texts is None:
    tiny_cfg.train_texts = 1024
if tiny_cfg.val_texts is None:
    tiny_cfg.val_texts = 256

train_loader, _ = build_dataloaders(
    model_config=sample_model_config,
    data_config=tiny_cfg,
    batch_size=8,
    num_workers=0,
    seed=2026,
    distributed=False,
    verbose=True,
)
inputs, targets = next(iter(train_loader))
source = 'tinystories_train_loader'

# 重构 full sequence: z = [prefix || suffix], z 长度 = block_size + 1
full_clean = torch.cat([inputs, targets[:, -1:].clone()], dim=1).to(device=device, dtype=torch.long)

full_len = full_clean.size(1)
prefix_len = max(4, int(full_len * 0.50))
if prefix_len >= full_len:
    prefix_len = full_len - 1
suffix_len = full_len - prefix_len
suffix_start = prefix_len

print('sample_source:', source)
print('batch_size:', full_clean.size(0))
print('full_len:', full_len, 'prefix_len:', prefix_len, 'suffix_len:', suffix_len)
print('suffix_start_index(in z):', suffix_start)
print('example clean prefix tokens:', full_clean[0, :prefix_len].tolist())
print('example clean suffix tokens:', full_clean[0, prefix_len:].tolist())


In [ ]:
# Cell 4. 构造更强 corruption（保持 target suffix 不变）
def _mutate_values(orig: torch.Tensor, vocab_size: int, delta: int = 17) -> torch.Tensor:
    # 保证替换后不等于原值
    return (orig + delta) % max(2, vocab_size)


def build_corruption_candidates(full_tokens: torch.Tensor, prefix_len: int, vocab_size: int):
    cands = {}
    meta = {}

    # 候选1：边界关键位点 + 邻域多点扰动
    c1 = full_tokens.clone()
    pos1 = sorted(set([max(1, prefix_len - 1), max(1, prefix_len - 2), max(1, prefix_len - 3)]))
    old1 = c1[:, pos1].clone()
    c1[:, pos1] = _mutate_values(c1[:, pos1], vocab_size, delta=11)
    cands['boundary_window_flip'] = c1
    meta['boundary_window_flip'] = {'positions': pos1, 'old_sample0': old1[0].tolist(), 'new_sample0': c1[0, pos1].tolist()}

    # 候选2：prefix 后半段随机化（更强）
    c2 = full_tokens.clone()
    start2 = max(1, prefix_len // 2)
    pos2 = list(range(start2, prefix_len))
    if pos2:
        torch.manual_seed(123)
        rand = torch.randint(0, vocab_size, (c2.size(0), len(pos2)), device=c2.device)
        # 避免恰好等于原值
        eq = rand == c2[:, pos2]
        rand = torch.where(eq, _mutate_values(rand, vocab_size, delta=7), rand)
        old2 = c2[:, pos2].clone()
        c2[:, pos2] = rand
        cands['prefix_tail_randomize'] = c2
        meta['prefix_tail_randomize'] = {
            'positions': [pos2[0], pos2[-1], len(pos2)],
            'old_sample0_head': old2[0, :min(6, len(pos2))].tolist(),
            'new_sample0_head': c2[0, pos2][:min(6, len(pos2))].tolist(),
        }

    # 候选3：prefix 后半段逆序（结构性破坏）
    c3 = full_tokens.clone()
    start3 = max(1, prefix_len // 2)
    pos3 = list(range(start3, prefix_len))
    if len(pos3) >= 2:
        old3 = c3[:, pos3].clone()
        c3[:, pos3] = torch.flip(c3[:, pos3], dims=[1])
        cands['prefix_tail_reverse'] = c3
        meta['prefix_tail_reverse'] = {
            'positions': [pos3[0], pos3[-1], len(pos3)],
            'old_sample0_head': old3[0, :min(6, len(pos3))].tolist(),
            'new_sample0_head': c3[0, pos3][:min(6, len(pos3))].tolist(),
        }

    # 强约束：suffix 保持不变
    for name, cand in cands.items():
        assert torch.equal(cand[:, prefix_len:], full_tokens[:, prefix_len:]), f'{name} 改动了 suffix，非法。'

    return cands, meta


corruption_candidates, corruption_meta = build_corruption_candidates(
    full_tokens=full_clean,
    prefix_len=prefix_len,
    vocab_size=experiment.model.vocab_size,
)

print('corruption candidate names:', list(corruption_candidates.keys()))
for name in corruption_candidates:
    info = corruption_meta[name]
    print(f"\n[{name}] info:")
    print(json.dumps(info, ensure_ascii=False, indent=2))
    diff_positions = (corruption_candidates[name][0, :prefix_len] != full_clean[0, :prefix_len]).nonzero(as_tuple=False).view(-1).tolist()
    print(f"[{name}] sample0 changed positions in prefix:", diff_positions)

print('suffix unchanged check:', all(torch.equal(v[:, prefix_len:], full_clean[:, prefix_len:]) for v in corruption_candidates.values()))


In [ ]:
# Cell 5. scoring helper（notebook glue code）
def suffix_metrics_from_logits(logits: torch.Tensor, target_full_tokens: torch.Tensor, prefix_len: int, topk: int = 5):
    # logits: [B, L-1, V], target_full_tokens: [B, L]
    targets = target_full_tokens[:, 1:]  # [B, L-1]
    start = prefix_len - 1
    if start < 0 or start >= targets.size(1):
        raise ValueError(f'非法 prefix_len={prefix_len} for targets_len={targets.size(1)}')

    suffix_targets = targets[:, start:]   # [B, S]
    suffix_logits = logits[:, start:, :]  # [B, S, V]

    log_probs = torch.log_softmax(suffix_logits, dim=-1)
    per_pos_logprob = torch.gather(log_probs, -1, suffix_targets.unsqueeze(-1)).squeeze(-1)  # [B, S]

    suffix_sum_logprob = per_pos_logprob.sum(dim=1)
    suffix_avg_logprob = per_pos_logprob.mean(dim=1)

    pred_next = suffix_logits.argmax(dim=-1)
    exact_hit_per_sample = (pred_next == suffix_targets).float().mean(dim=1)

    k = int(min(topk, suffix_logits.size(-1)))
    topk_ids = torch.topk(suffix_logits, k=k, dim=-1).indices
    topk_hit_per_sample = (topk_ids == suffix_targets.unsqueeze(-1)).any(dim=-1).float().mean(dim=1)

    per_pos_topk_hit = (topk_ids == suffix_targets.unsqueeze(-1)).any(dim=-1).float()  # [B,S]

    target_logits = torch.gather(suffix_logits, -1, suffix_targets.unsqueeze(-1))
    rank = (suffix_logits > target_logits).sum(dim=-1) + 1  # [B,S], 1=best
    avg_rank_per_sample = rank.float().mean(dim=1)

    return {
        'suffix_sum_logprob': float(suffix_sum_logprob.mean().item()),
        'suffix_avg_logprob': float(suffix_avg_logprob.mean().item()),
        'per_position_logprob_curve': per_pos_logprob.mean(dim=0).detach().cpu(),
        'exact_hit': float(exact_hit_per_sample.mean().item()),
        'topk_hit': float(topk_hit_per_sample.mean().item()),
        'per_position_topk_hit_curve': per_pos_topk_hit.mean(dim=0).detach().cpu(),
        'avg_rank': float(avg_rank_per_sample.mean().item()),
        'rank_tensor': rank.detach().cpu(),
    }


def run_condition(current_input_ids: torch.Tensor, target_full_tokens: torch.Tensor):
    with torch.no_grad():
        outputs = model(current_input_ids, return_cache=True, return_intermediates=True)
    metrics = suffix_metrics_from_logits(
        logits=outputs['logits'],
        target_full_tokens=target_full_tokens,
        prefix_len=prefix_len,
        topk=5,
    )
    return outputs, metrics


def print_metrics(title: str, metrics: dict):
    print(f"[{title}] suffix_sum_logprob = {metrics['suffix_sum_logprob']:.6f}")
    print(f"[{title}] suffix_avg_logprob = {metrics['suffix_avg_logprob']:.6f}")
    print(f"[{title}] exact_hit = {metrics['exact_hit']:.4f}")
    print(f"[{title}] topk_hit = {metrics['topk_hit']:.4f}")
    print(f"[{title}] avg_rank = {metrics['avg_rank']:.3f}")


In [ ]:
# Cell 6. clean / corrupted 基线比较（先保证 corrupted 有可见 drop）
clean_input_ids = full_clean[:, :-1]
clean_outputs, clean_metrics = run_condition(clean_input_ids, target_full_tokens=full_clean)
print_metrics('clean', clean_metrics)

candidate_results = {}
for name, full_cor in corruption_candidates.items():
    cor_input_ids = full_cor[:, :-1]
    _, cor_metrics = run_condition(cor_input_ids, target_full_tokens=full_clean)  # suffix target 固定为 clean suffix
    corrupted_drop = clean_metrics['suffix_avg_logprob'] - cor_metrics['suffix_avg_logprob']
    candidate_results[name] = {
        'metrics': cor_metrics,
        'corrupted_drop': float(corrupted_drop),
        'full_corrupted': full_cor,
    }

# 选择 drop 最大的 corruption
selected_corruption_name = max(candidate_results.keys(), key=lambda k: candidate_results[k]['corrupted_drop'])
selected = candidate_results[selected_corruption_name]
corrupted_full = selected['full_corrupted']
corrupted_input_ids = corrupted_full[:, :-1]
corrupted_metrics = selected['metrics']

print('\ncorruption candidates by drop (clean - corrupted):')
for name, payload in sorted(candidate_results.items(), key=lambda kv: kv[1]['corrupted_drop'], reverse=True):
    print(f"- {name}: drop={payload['corrupted_drop']:+.6f}")

print(f"\nselected_corruption: {selected_corruption_name}")
print('clean prefix sample0:', full_clean[0, :prefix_len].tolist())
print('corrupted prefix sample0:', corrupted_full[0, :prefix_len].tolist())
print('changed positions sample0:', (corrupted_full[0, :prefix_len] != full_clean[0, :prefix_len]).nonzero(as_tuple=False).view(-1).tolist())

print_metrics('corrupted(selected)', corrupted_metrics)
if selected['corrupted_drop'] < 0.01:
    print('[WARNING] 当前样本/扰动仍偏弱（drop < 0.01），后续 patch 信号可能仍有限。')


In [ ]:
# Cell 7. patched 条件比较（site sweep -> 选 recovery gain 最大）
clean_cache = clean_outputs['cache']
cache_keys = sorted(clean_cache.data.keys())

preferred_sites = [
    'blocks.0.attn_out',
    'blocks.0.output',
    'blocks.1.attn_out',
    'blocks.1.output',
    'embedding_out',
]
available_sites = [s for s in preferred_sites if s in clean_cache.data]
if not available_sites:
    # 回退：从 cache 中选若干稳定位点
    available_sites = [k for k in cache_keys if k.endswith(('attn_out', 'mlp_out', 'output'))][:6]
if not available_sites:
    raise RuntimeError('未找到可 patch 的位点。')

site_results = {}
for site in available_sites:
    overrides = patch_from_cache(clean_cache, site)
    with torch.no_grad():
        patched_outputs = model(
            corrupted_input_ids,
            return_cache=True,
            return_intermediates=True,
            activation_overrides=overrides,
        )
    patched_metrics = suffix_metrics_from_logits(
        logits=patched_outputs['logits'],
        target_full_tokens=full_clean,
        prefix_len=prefix_len,
        topk=5,
    )
    recovery_gain = patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob']
    site_results[site] = {
        'patched_outputs': patched_outputs,
        'patched_metrics': patched_metrics,
        'recovery_gain': float(recovery_gain),
    }

selected_site = max(site_results.keys(), key=lambda s: site_results[s]['recovery_gain'])
patched_outputs = site_results[selected_site]['patched_outputs']
patched_metrics = site_results[selected_site]['patched_metrics']

clean_minus_corrupted = clean_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob']
patched_minus_corrupted = patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob']
patched_minus_clean = patched_metrics['suffix_avg_logprob'] - clean_metrics['suffix_avg_logprob']

if abs(clean_minus_corrupted) > 1e-8:
    recovery_ratio = patched_minus_corrupted / clean_minus_corrupted
else:
    recovery_ratio = float('nan')

print('patch site recovery_gain ranking:')
for s, payload in sorted(site_results.items(), key=lambda kv: kv[1]['recovery_gain'], reverse=True):
    print(f"- {s:24s} recovery_gain={payload['recovery_gain']:+.6f}")

print(f"\nselected_patch_site: {selected_site}")
print_metrics('patched(selected_site)', patched_metrics)

delta_report = {
    'clean_minus_corrupted': {
        'suffix_avg_logprob': clean_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
        'suffix_sum_logprob': clean_metrics['suffix_sum_logprob'] - corrupted_metrics['suffix_sum_logprob'],
        'exact_hit': clean_metrics['exact_hit'] - corrupted_metrics['exact_hit'],
        'topk_hit': clean_metrics['topk_hit'] - corrupted_metrics['topk_hit'],
        'avg_rank': clean_metrics['avg_rank'] - corrupted_metrics['avg_rank'],
    },
    'patched_minus_corrupted': {
        'suffix_avg_logprob': patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
        'suffix_sum_logprob': patched_metrics['suffix_sum_logprob'] - corrupted_metrics['suffix_sum_logprob'],
        'exact_hit': patched_metrics['exact_hit'] - corrupted_metrics['exact_hit'],
        'topk_hit': patched_metrics['topk_hit'] - corrupted_metrics['topk_hit'],
        'avg_rank': patched_metrics['avg_rank'] - corrupted_metrics['avg_rank'],
    },
    'patched_minus_clean': {
        'suffix_avg_logprob': patched_metrics['suffix_avg_logprob'] - clean_metrics['suffix_avg_logprob'],
        'suffix_sum_logprob': patched_metrics['suffix_sum_logprob'] - clean_metrics['suffix_sum_logprob'],
        'exact_hit': patched_metrics['exact_hit'] - clean_metrics['exact_hit'],
        'topk_hit': patched_metrics['topk_hit'] - clean_metrics['topk_hit'],
        'avg_rank': patched_metrics['avg_rank'] - clean_metrics['avg_rank'],
    },
    'recovery_ratio': recovery_ratio,
}
print(json.dumps(delta_report, indent=2, ensure_ascii=False))


In [ ]:
# Cell 8. 可视化
conditions = ['clean', 'corrupted', 'patched']
metrics_by_condition = {
    'clean': clean_metrics,
    'corrupted': corrupted_metrics,
    'patched': patched_metrics,
}

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# 1) suffix avg logprob
axes[0, 0].bar(conditions, [metrics_by_condition[c]['suffix_avg_logprob'] for c in conditions])
axes[0, 0].set_title('Suffix Avg Logprob')
axes[0, 0].set_ylabel('logprob')

# 2) suffix sum logprob
axes[0, 1].bar(conditions, [metrics_by_condition[c]['suffix_sum_logprob'] for c in conditions])
axes[0, 1].set_title('Suffix Sum Logprob')
axes[0, 1].set_ylabel('logprob')

# 3) per-position suffix token logprob
x = list(range(len(clean_metrics['per_position_logprob_curve'])))
for c in conditions:
    axes[0, 2].plot(x, metrics_by_condition[c]['per_position_logprob_curve'].tolist(), marker='o', label=c)
axes[0, 2].set_title('Per-position Suffix Token Logprob')
axes[0, 2].set_xlabel('suffix position')
axes[0, 2].set_ylabel('logprob')
axes[0, 2].legend()

# 4) exact hit / top-k hit
bar_x = list(range(len(conditions)))
w = 0.35
axes[1, 0].bar([i - w/2 for i in bar_x], [metrics_by_condition[c]['exact_hit'] for c in conditions], width=w, label='exact_hit')
axes[1, 0].bar([i + w/2 for i in bar_x], [metrics_by_condition[c]['topk_hit'] for c in conditions], width=w, label='topk_hit')
axes[1, 0].set_xticks(bar_x)
axes[1, 0].set_xticklabels(conditions)
axes[1, 0].set_ylim(0.0, 1.0)
axes[1, 0].set_title('Exact Hit vs Top-k Hit')
axes[1, 0].legend()

# 5) avg rank
axes[1, 1].bar(conditions, [metrics_by_condition[c]['avg_rank'] for c in conditions])
axes[1, 1].set_title('Average Target Token Rank (lower is better)')
axes[1, 1].set_ylabel('rank')

# 6) drop / recovery
drop_recovery_labels = ['clean-corrupted', 'patched-corrupted', 'patched-clean']
drop_recovery_values = [
    clean_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
    patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
    patched_metrics['suffix_avg_logprob'] - clean_metrics['suffix_avg_logprob'],
]
axes[1, 2].bar(drop_recovery_labels, drop_recovery_values)
axes[1, 2].axhline(0.0, color='black', linewidth=1)
axes[1, 2].set_title('Drop / Recovery Gain (suffix avg logprob)')
axes[1, 2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

summary_table = [
    {
        'condition': c,
        'suffix_sum_logprob': metrics_by_condition[c]['suffix_sum_logprob'],
        'suffix_avg_logprob': metrics_by_condition[c]['suffix_avg_logprob'],
        'exact_hit': metrics_by_condition[c]['exact_hit'],
        'topk_hit': metrics_by_condition[c]['topk_hit'],
        'avg_rank': metrics_by_condition[c]['avg_rank'],
    }
    for c in conditions
]
print('\ncondition summary:')
print(json.dumps(summary_table, indent=2, ensure_ascii=False))

delta_table = {
    'clean_minus_corrupted': clean_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
    'patched_minus_corrupted': patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
    'patched_minus_clean': patched_metrics['suffix_avg_logprob'] - clean_metrics['suffix_avg_logprob'],
}
print('\ndelta summary (suffix_avg_logprob):')
print(json.dumps(delta_table, indent=2, ensure_ascii=False))


## Cell 9. 结果解释

本 revised notebook 与上一版相比，增强点是：
1. 在同一批样本上先做 corruption 候选比较，优先选择“clean->corrupted drop 最大”的扰动；
2. patched 阶段不再固定单一 site，而是对少量稳定位点做 sweep，选择 recovery gain 最大位点；
3. 指标增加了 `suffix_sum_logprob`、`avg_rank`、差值与 recovery_ratio，并在图中集中展示。

这仍然是 Prefix-conditioned target suffix 的 teacher-forced scoring：
- 评分始终基于 logits + 固定 suffix target；
- 没有做 generate/sample，也不是完整 extraction attack。

如果这版仍然信号弱，最可能原因通常是：
- 当前 checkpoint 对该样本簇本身不确定；
- corruption 仍未命中真正决定 suffix 的条件位；
- 所选 patch site 对整段 suffix 影响不足。
